# Calculate Errors

Calculate MAE and RMSE for the two prediction rules.

Open this notebook in `notebooks/M4/`. Run the cells in order. Saving again replaces this notebook's output files.


## Shared folder setup

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

## The five-row metric example

In [ ]:
from pathlib import Path
import pandas as pd
example = pd.DataFrame({
    "actual_kwh": [1.0, 0.5, 0.8, float("nan"), 0.9],
    "pred_prev_hour": [0.6, 0.5, 1.2, 0.4, float("nan")],
    "pred_prev_day": [0.8, 0.2, 0.8, 0.3, 0.7]})
example

## Find missing actual values

In [ ]:
example_missing_target = example["actual_kwh"].isna()
example_missing_target

## Find missing predictions without double-counting

In [ ]:
example_missing_prediction = ~example_missing_target & example[["pred_prev_hour",
    "pred_prev_day"]].isna().any(axis=1)
example_missing_prediction

## Keep a common comparison set

In [ ]:
example_valid = ~example_missing_target & ~example_missing_prediction
example_used = example[example_valid]
example_used

## Calculate signed errors

In [ ]:
example_errors = pd.DataFrame({
    "previous_hour": example_used["actual_kwh"] - example_used["pred_prev_hour"],
    "previous_day": example_used["actual_kwh"] - example_used["pred_prev_day"]})
example_errors

## Remove the signs

In [ ]:
example_absolute = example_errors.abs()
example_absolute

## Calculate MAE

In [ ]:
example_mae = example_absolute.mean()
example_mae

## Square the errors

In [ ]:
example_squared = example_errors ** 2
example_squared

## Calculate RMSE

In [ ]:
example_rmse = example_squared.mean() ** 0.5
example_rmse

## Collect the hand-example results

In [ ]:
example_scores = pd.DataFrame({"mae_kwh": example_mae, "rmse_kwh": example_rmse})
example_scores.index.name = "rule"
example_scores["n_total"] = len(example)
example_scores["n_missing_target"] = example_missing_target.sum()
example_scores["n_missing_prediction"] = example_missing_prediction.sum()
example_scores["n_scored"] = example_valid.sum()
example_scores

## Choose the predictions

In [ ]:
prediction_path = PROJECT_ROOT / "results/baseline_predictions_sample.csv"
predictions = pd.read_csv(prediction_path)
predictions["target_hour"] = pd.to_datetime(predictions["target_hour"], utc=True)
predictions.head()

## See the supplied scope

In [ ]:
predictions["LCLid"].unique(), len(predictions), predictions["target_hour"].min(), predictions["target_hour"].max()

## Count missing actual values

In [ ]:
missing_target = predictions["actual_kwh"].isna()
missing_target.sum()

## Count rows missing either prediction

In [ ]:
missing_prediction = ~missing_target & predictions[["pred_prev_hour",
    "pred_prev_day"]].isna().any(axis=1)
missing_prediction.sum()

## Use the same rows for both baselines

In [ ]:
valid = ~missing_target & ~missing_prediction
used = predictions[valid]
len(used), used.head()

## Show the scoring counts

In [ ]:
row_counts = pd.Series({"n_total": len(predictions), "n_missing_target": missing_target.sum(),
                        "n_missing_prediction": missing_prediction.sum(), "n_scored": valid.sum()})
row_counts

## Calculate both errors on the common rows

In [ ]:
errors = pd.DataFrame({
    "previous_hour": used["actual_kwh"] - used["pred_prev_hour"],
    "previous_day": used["actual_kwh"] - used["pred_prev_day"]})
errors.head()

## Calculate the prototype MAE

In [ ]:
mae_values = errors.abs().mean()
mae_values

## Calculate the prototype RMSE

In [ ]:
rmse_values = (errors ** 2).mean() ** 0.5
rmse_values

## Build the score table

In [ ]:
scores = pd.DataFrame({"mae_kwh": mae_values, "rmse_kwh": rmse_values})
scores.index.name = "rule"
scores = scores.reset_index()
for column, value in row_counts.items():
    scores[column] = value
scores

## Keep the task and split labels

In [ ]:
scores.insert(0, "task_id", "M4-W07-T02")
scores.insert(1, "split", "prototype")
scores

## Read the results schema

In [ ]:
schema = pd.read_csv(PROJECT_ROOT / "notebooks/M4/score_columns.csv")
schema

## Order the columns

In [ ]:
scores = scores[schema["column"]].copy()
scores

## Compare the row counts

In [ ]:
row_counts["n_total"] == row_counts["n_missing_target"] + row_counts["n_missing_prediction"] + row_counts["n_scored"]

## Choose the output folder

In [ ]:
output_dir = PROJECT_ROOT / 'results'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

## Save the prototype scores

In [ ]:
scores_path = output_dir / "baseline_scores_sample.csv"
scores.to_csv(scores_path, index=False)
scores_path

## Save the hand-example scores

In [ ]:
example_path = output_dir / "error_calculation_example.csv"
example_scores.reset_index().to_csv(example_path, index=False)
example_path

## Save the column descriptions

In [ ]:
schema_path = output_dir / "score_columns.csv"
schema.to_csv(schema_path, index=False)
schema_path

## Read back the saved score table

In [ ]:
pd.read_csv(scores_path)